# Orlando theme park wait-time collector
# Pulls parks, attractions and live wait times for the Orlando resorts from ThemeParks.wiki and writes them to Delta tables in the attached lakehouse. Pure Python, no Spark.
# **Tables written (default lakehouse)**
- `dim_entity`: parks, attractions, shows and restaurants. Overwritten each run.
- `fact_wait_snapshot`: one row per attraction per run. Appended each run.
# **Setup**
1. Attach a lakehouse to this notebook.
2. Run it once by hand to confirm both tables appear.
3. To collect history, schedule this notebook hourly in your own workspace.
4. If you build a pipeline that runs collection, reporting-table preparation and model refresh, schedule that pipeline instead of scheduling the collector separately.


In [ ]:
import json
import time
import urllib.request

import pandas as pd
from deltalake import DeltaTable, write_deltalake

import notebookutils

DIM_TABLE = "dim_entity"
FACT_TABLE = "fact_wait_snapshot"

BASE = "https://api.themeparks.wiki/v1/entity/"
DESTINATIONS = {
    "e957da41-3552-4cf6-b636-5babc5cbc4e5": "Walt Disney World Resort",
    "89db5d43-c434-4097-b71f-f6869f495a22": "Universal Orlando Resort",
    "643e837e-b244-4663-8d3a-148c26ecba9c": "SeaWorld Parks and Resorts Orlando",
}

# --- OneLake connection ---
# The /lakehouse/default mount can't rename files, which Delta commits need, so write through the OneLake abfss path.
ctx = notebookutils.runtime.context
lakehouse_id = ctx.get("defaultLakehouseId")
if not lakehouse_id:
    raise RuntimeError("No default lakehouse is attached to this notebook. Attach one and rerun.")
workspace_id = ctx.get("defaultLakehouseWorkspaceId") or ctx["currentWorkspaceId"]

# Schema-enabled lakehouse: tables live under Tables/<schema>. Clear SCHEMA for a lakehouse without schemas.
SCHEMA = "dbo"
TABLES_ROOT = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables" + (f"/{SCHEMA}" if SCHEMA else "")
STORAGE_OPTIONS = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}
# --- end OneLake connection ---

In [ ]:
def get(url, retries=3):
    """GET a JSON document, retrying a couple of times on transient errors."""
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "fabric101-demo/0.1"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                return json.load(resp)
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 * (attempt + 1))

In [ ]:
# Dimension: every park, attraction, show and restaurant, linked to its park.
dim_rows = []
for dest_id, dest_name in DESTINATIONS.items():
    for e in get(BASE + dest_id + "/children").get("children", []):
        loc = e.get("location") or {}
        dim_rows.append({
            "entity_id": e.get("id"),
            "entity_name": e.get("name"),
            "entity_type": e.get("entityType"),
            "destination": dest_name,
            "latitude": loc.get("latitude"),
            "longitude": loc.get("longitude"),
            "timezone": e.get("timezone"),
        })

# The resort-level call has no park link, so ask each park for its own children.
parks = {r["entity_id"]: r["entity_name"] for r in dim_rows if r["entity_type"] == "PARK"}
park_of = {}
for park_id in parks:
    for child in get(BASE + park_id + "/children").get("children", []):
        park_of[child["id"]] = park_id

for r in dim_rows:
    r["park_id"] = r["entity_id"] if r["entity_type"] == "PARK" else park_of.get(r["entity_id"])
    r["park_name"] = parks.get(r["park_id"])

df_dim = pd.DataFrame(dim_rows)
for col in ["entity_id", "entity_name", "entity_type", "destination", "timezone", "park_id", "park_name"]:
    df_dim[col] = df_dim[col].astype("string")
for col in ["latitude", "longitude"]:
    df_dim[col] = pd.to_numeric(df_dim[col], errors="coerce")

print(f"{len(df_dim):,} entities, {len(parks)} parks")

In [ ]:
# Fact: one row per attraction right now. Everything outside the parks' hours shows up as CLOSED with no wait.
snapshot_utc = pd.Timestamp.now(tz="UTC").floor("s")

fact_rows = []
for dest_id in DESTINATIONS:
    for e in get(BASE + dest_id + "/live").get("liveData", []):
        if e.get("entityType") != "ATTRACTION":
            continue
        standby = ((e.get("queue") or {}).get("STANDBY") or {}).get("waitTime")
        fact_rows.append({
            "snapshot_utc": snapshot_utc,
            "entity_id": e.get("id"),
            "park_id": e.get("parkId"),
            "status": e.get("status"),
            "standby_wait_min": standby,
            "last_updated_utc": e.get("lastUpdated"),
        })

df_fact = pd.DataFrame(fact_rows)
for col in ["entity_id", "park_id", "status"]:
    df_fact[col] = df_fact[col].astype("string")
df_fact["snapshot_utc"] = pd.to_datetime(df_fact["snapshot_utc"], utc=True).astype("datetime64[us, UTC]")
df_fact["last_updated_utc"] = pd.to_datetime(df_fact["last_updated_utc"], utc=True, errors="coerce").astype("datetime64[us, UTC]")
df_fact["standby_wait_min"] = pd.to_numeric(df_fact["standby_wait_min"], errors="coerce").astype("Int64")

print(f"{len(df_fact):,} attractions at {snapshot_utc}")

In [ ]:
write_deltalake(f"{TABLES_ROOT}/{DIM_TABLE}", df_dim, mode="overwrite", schema_mode="overwrite", storage_options=STORAGE_OPTIONS)
write_deltalake(f"{TABLES_ROOT}/{FACT_TABLE}", df_fact, mode="append", storage_options=STORAGE_OPTIONS)

In [ ]:
# Sanity check: how much history have we collected, and what are the longest waits in the latest snapshot?
fact = DeltaTable(f"{TABLES_ROOT}/{FACT_TABLE}", storage_options=STORAGE_OPTIONS).to_pandas()
latest = fact[fact["snapshot_utc"] == fact["snapshot_utc"].max()]
print(f"{len(fact):,} rows across {fact['snapshot_utc'].nunique()} snapshots, latest {fact['snapshot_utc'].max()}")

top = (
    latest.merge(df_dim[["entity_id", "entity_name", "park_name"]], on="entity_id", how="left")
    .dropna(subset=["standby_wait_min"])
    .nlargest(10, "standby_wait_min")[["park_name", "entity_name", "standby_wait_min"]]
)
print(top.to_string(index=False))